In [4]:
!pip install rasterio
import rasterio
import numpy as np
from glob import glob
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer
from sklearn.model_selection import train_test_split


# Load all TIFF images and masks
image_paths = sorted(glob("/kaggle/input/cloud-masking-dataset/content/train/data/*.tif"))
mask_paths = sorted(glob("/kaggle/input/cloud-masking-dataset/content/train/masks/*.tif"))

# Load a single TIFF image to check shape
with rasterio.open(image_paths[0]) as src:
    sample_img = src.read()  # Shape: (C, H, W)
print(f"Image shape: {sample_img.shape}")  # Should be (4, H, W) for 4-channel

Image shape: (4, 512, 512)


In [5]:
def load_tiff_as_features(image_paths, mask_paths):
    X, y = [], []
    for img_path, mask_path in zip(image_paths, mask_paths):
        with rasterio.open(img_path) as src:
            img = src.read().transpose(1, 2, 0)  # Convert (C, H, W) → (H, W, C)
        with rasterio.open(mask_path) as src:
            mask = src.read(1)  # Read first band (H, W)
        
        # Flatten
        X.append(img.reshape(-1, img.shape[-1]))  # (H*W, 4)
        y.append(mask.reshape(-1))                # (H*W,)
    
    return np.vstack(X), np.concatenate(y)

X, y = load_tiff_as_features(image_paths, mask_paths)  # Use subset for testing

In [9]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [11]:
# 1. Define Dice Score function (modified for GridSearchCV)
def dice_score_grid(y_true, y_pred):
    y_true = y_true.astype(bool)
    y_pred = y_pred.astype(bool)
    intersection = np.logical_and(y_true, y_pred).sum()
    union = y_true.sum() + y_pred.sum()
    return (2. * intersection + 1e-7) / (union + 1e-7)

# 2. Create a scorer (higher_is_better=True since we want to maximize Dice)
dice_scorer = make_scorer(dice_score_grid, greater_is_better=True)

# 3. Define parameter grid
param_grid = {
    'n_estimators': [50,100, 200],
    'max_depth': [10, 15, None],
    'min_samples_split': [2, 5],
    'class_weight': ['balanced', None],
}

# 4. Initialize GridSearchCV
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(n_jobs=-1, random_state=42),
    param_grid=param_grid,
    scoring=dice_scorer,
    cv=3,  # 3-fold cross-validation
    verbose=2,
    n_jobs=-1,  # Parallel processing
)

# 5. Run grid search
grid_search.fit(X_train, y_train)

# 6. Get best results
print("Best Parameters:", grid_search.best_params_)
print("Best Dice Score:", grid_search.best_score_)

# 7. Evaluate on test set
best_model = grid_search.best_estimator_
y_pred_test = best_model.predict(X_test)
test_dice = dice_score_grid(y_test, y_pred_test)
print(f"Test Dice Score: {test_dice:.4f}")

Fitting 3 folds for each of 36 candidates, totalling 108 fits
[CV] END class_weight=balanced, max_depth=10, min_samples_split=2, n_estimators=50; total time= 4.1min
[CV] END class_weight=balanced, max_depth=10, min_samples_split=2, n_estimators=100; total time= 7.6min
[CV] END class_weight=balanced, max_depth=10, min_samples_split=2, n_estimators=200; total time=14.2min


/usr/local/lib/python3.11/dist-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END class_weight=balanced, max_depth=10, min_samples_split=2, n_estimators=50; total time= 4.1min
[CV] END class_weight=balanced, max_depth=10, min_samples_split=2, n_estimators=100; total time= 8.0min
[CV] END class_weight=balanced, max_depth=10, min_samples_split=5, n_estimators=50; total time= 3.6min
[CV] END class_weight=balanced, max_depth=10, min_samples_split=5, n_estimators=50; total time= 3.4min
[CV] END class_weight=balanced, max_depth=10, min_samples_split=5, n_estimators=100; total time= 6.5min
[CV] END class_weight=balanced, max_depth=10, min_samples_split=5, n_estimators=200; total time=14.7min
[CV] END class_weight=balanced, max_depth=10, min_samples_split=2, n_estimators=100; total time= 8.8min
[CV] END class_weight=balanced, max_depth=10, min_samples_split=2, n_estimators=200; total time=14.8min
[CV] END class_weight=balanced, max_depth=10, min_samples_split=5, n_estimators=100; total time= 6.6min
[CV] END class_weight=balanced, max_depth=15, min_samples_split=2, 

In [ ]:
def predict_and_save_mask(model, tiff_path, output_path):
    with rasterio.open(tiff_path) as src:
        img = src.read().transpose(1, 2, 0)  # (H, W, C)
        meta = src.meta
    
    # Predict
    pixels = img.reshape(-1, img.shape[-1])
    pred = model.predict(pixels).reshape(img.shape[:2])  # (H, W)
    
    # Save as TIFF with original metadata
    meta.update(count=1, dtype='uint8')
    with rasterio.open(output_path, 'w', **meta) as dst:
        dst.write(pred.astype(np.uint8), 1)

# Example usage
# predict_and_save_mask(clf, "cloud-masking-dataset/test/data/image_123.tif", "predicted_mask.tif")